# ContractRisk M3 — integración 3B/v5 y calibración del juez

Sesión nueva de Colab con GPU T4. Ejecuta cuatro variantes sobre ocho preguntas REALES DE DESARROLLO (32 respuestas), después 12 pares sintéticos breves/extensos antes/después de la mitigación de verbosidad (48 juicios). No usa el gold final ni calcula aún RAGAS. El mismo modelo 3B actúa en llamadas separadas como generador y juez: esa dependencia debe declararse. Una corrida completada no certifica calidad. Devuelva todos los resultados, incluso fallos.


In [1]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, os, sys, subprocess
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Suba únicamente contractrisk_m3_integracion_v5_codigo.zip')
archive = next(iter(uploaded))
ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_m3_integracion_v5_', dir='/content'))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Ruta no permitida en ZIP')
    z.extractall(ROOT)
os.chdir(ROOT)
assert Path('m3/run_integration_v5.py').is_file(), 'ZIP incorrecto'
print(ROOT)


Saving contractrisk_m3_integracion_v5_codigo.zip to contractrisk_m3_integracion_v5_codigo.zip
/content/contractrisk_m3_integracion_v5_qugxv2ex


In [2]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 28.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 75.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.2/974.2 kB 47.8 MB

In [3]:
# La carpeta debe ser nueva: no borre resultados anteriores para repetir.
run_visible([sys.executable, '-u', 'm3/run_integration_v5.py', '--output', 'integracion_v5_resultados'], 'execution.log')


I0000 00:00:1793257562.555616    1022 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793257565.244427    1022 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Building dense index...
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Loading fixed-revision reranker...

Fetching 7 files: 100%|██████████| 7/7 [00:05<00:00,  1.33it/s]
L

In [4]:
# Ejecute también tras un fallo para conservar diagnóstico.
import json, shutil
result_dir = ROOT / 'integracion_v5_resultados'
result_dir.mkdir(exist_ok=True)
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8') if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_integracion_v5_resultados'), 'zip', result_dir)
files.download(bundle)


{
  "status": "completed",
  "purpose": "integration_v5_and_calibration_not_final_scorecard",
  "started_utc": "2026-10-29T07:05:49.681474+00:00",
  "seed": 42,
  "python": "3.13.16",
  "models": {
    "embedding": [
      "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    ],
    "generator": [
      "Qwen/Qwen2.5-3B-Instruct",
      "aa8e72537993ba99e69dfaafa59ed015b17504d1"
    ],
    "reranker": [
      "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "1427fd652930e4ba29e8149678df786c240d8825"
    ]
  },
  "sqlite": {
    "backend": "sqlite3",
    "sqlite_version": "3.45.1",
    "fts5_checked": true
  },
  "gpu": "Tesla T4",
  "packages": {
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "transformers": "4.57.6",
    "sentence-transformers": "3.4.1",
    "chromadb": "0.6.3",
    "rank-bm25": "0.2.2",
    "pysqlite3-binary": "0.5.4.post2"
  },
  "source_hashes": {
    "freeze_gold.py": "f78b353ebcc66dc1f9a7617

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Devuelva el ZIP y este notebook descargado con sus salidas. No sustituya las respuestas incorrectas ni los errores. La revisión separará funcionamiento técnico, recuperación, generación y decisiones de herramienta.